<a href="https://colab.research.google.com/github/chintu4/LLM-based-Movie-Recommendation/blob/main/text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [68]:
import pandas as pd

movies = pd.read_csv("movies_cleaned.csv")

In [69]:
movies.columns

Index(['Release_Date', 'Title', 'Overview', 'Popularity', 'Vote_Count',
       'Vote_Average', 'Original_Language', 'Genre', 'Poster_Url',
       'title_and_subtitle', 'tagged_description'],
      dtype='object')

In [70]:
movies["Genre"].value_counts().reset_index()

,Genre,count
0,Drama,374
1,Comedy,328
2,Horror,203
3,"Drama, Romance",194
4,"Horror, Thriller",161
...,...,...
2085,"Adventure, Fantasy, Drama, Mystery",1
2086,"Science Fiction, Comedy, Fantasy",1
2087,"Thriller, Science Fiction, Mystery, Horror",1
2088,"Family, Animation, Adventure, Fantasy, Science...",1


In [71]:
movies["Genre"].value_counts().reset_index().query("count > 50")

,Genre,count
0,Drama,374
1,Comedy,328
2,Horror,203
3,"Drama, Romance",194
4,"Horror, Thriller",161
5,"Comedy, Romance",160
6,"Comedy, Drama",107
7,Documentary,101
8,"Action, Thriller",98
9,"Comedy, Drama, Romance",86


In [72]:
movies[movies["Genre"] == "Juvenile Fiction"]

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url,title_and_subtitle,tagged_description


In [73]:
movies[movies["Genre"] == "Juvenile Nonfiction"]

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url,title_and_subtitle,tagged_description


In [74]:
import seaborn as sns
import matplotlib.pyplot as plt

In [75]:
import numpy as np

# Initialize 'simple_Genre' column. Default to 'Fiction' for simplicity.
movies['simple_Genre'] = 'Fiction'

# Define keywords that typically indicate a 'Nonfiction' genre.
nonfiction_keywords = [
    'documentary',
    'biography',
    'history',
    'science',
    'literary criticism',
    'philosophy',
    'religion',
    'juvenile nonfiction'
]

# Apply the classification: if any nonfiction keyword is in the original 'Genre', mark as 'Nonfiction'.
for keyword in nonfiction_keywords:
    movies.loc[movies['Genre'].str.contains(keyword, case=False, na=False), 'simple_Genre'] = 'Nonfiction'

# Now, filter the movies DataFrame to show only rows where 'simple_Genre' is not NaN.
# This line effectively replaces the original content of the cell.
movies[~(movies["simple_Genre"].isna())]

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url,title_and_subtitle,tagged_description,simple_Genre
0,2021-12-15,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4i...,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,Nonfiction
1,2022-03-01,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R3...,The Batman,"In his second year of fighting crime, Batman u...",Fiction
2,2022-02-25,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,Thriller,https://image.tmdb.org/t/p/original/vDHsLnOWKl...,No Exit,Stranded at a rest stop in the mountains durin...,Fiction
3,2021-11-24,Encanto,"The tale of an extraordinary family, the Madri...",2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",https://image.tmdb.org/t/p/original/4j0PNHkMr5...,Encanto,"The tale of an extraordinary family, the Madri...",Fiction
4,2021-12-22,The King's Man,As a collection of history's worst tyrants and...,1895.511,1793,7.0,en,"Action, Adventure, Thriller, War",https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...,The King's Man,As a collection of history's worst tyrants and...,Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...
8165,1973-10-15,Badlands,A dramatization of the Starkweather-Fugate kil...,13.357,896,7.6,en,"Drama, Crime",https://image.tmdb.org/t/p/original/z81rBzHNgi...,Badlands,A dramatization of the Starkweather-Fugate kil...,Fiction
8166,2020-10-01,Violent Delights,A female vampire falls in love with a man she ...,13.356,8,3.5,es,Horror,https://image.tmdb.org/t/p/original/4b6HY7rud6...,Violent Delights,A female vampire falls in love with a man she ...,Fiction
8167,2016-05-06,The Offering,When young and successful reporter Jamie finds...,13.355,94,5.0,en,"Mystery, Thriller, Horror",https://image.tmdb.org/t/p/original/h4uMM1wOhz...,The Offering,When young and successful reporter Jamie finds...,Fiction
8168,2021-03-31,The United States vs. Billie Holiday,Billie Holiday spent much of her career being ...,13.354,152,6.7,en,"Music, Drama, History",https://image.tmdb.org/t/p/original/vEzkxuE2sJ...,The United States vs. Billie Holiday,Billie Holiday spent much of her career being ...,Nonfiction


In [76]:
from transformers import pipeline

fiction_Genre = ["Fiction", "Nonfiction"]

pipe = pipeline("zero-shot-classification",
                model="facebook/bart-large-mnli",
                device=0)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [77]:
sequence = movies.loc[movies["simple_Genre"] == "Fiction", "Overview"].reset_index(drop=True)[0]

In [78]:
pipe(sequence, fiction_Genre)

{'sequence': 'In his second year of fighting crime, Batman uncovers corruption in Gotham City that connects to his own family while facing a serial killer known as the Riddler.',
 'labels': ['Nonfiction', 'Fiction'],
 'scores': [0.5836175084114075, 0.41638249158859253]}

In [79]:
import numpy as np

max_index = np.argmax(pipe(sequence, fiction_Genre)["scores"])
max_label = pipe(sequence, fiction_Genre)["labels"][max_index]
max_label

'Nonfiction'

In [80]:
def generate_predictions(sequence, Genre):
    predictions = pipe(sequence, Genre)
    max_index = np.argmax(predictions["scores"])
    max_label = predictions["labels"][max_index]
    return max_label

In [81]:
from tqdm import tqdm

actual_cats = []
predicted_cats = []

fiction_sequences = movies.loc[movies["simple_Genre"] == "Fiction", "Overview"].reset_index(drop=True)

for i in tqdm(range(0, len(fiction_sequences))):
    sequence = fiction_sequences[i]
    predicted_cats += [generate_predictions(sequence, fiction_Genre)]
    actual_cats += ["Fiction"]

100%|██████████| 6573/6573 [06:35<00:00, 16.61it/s]


In [ ]:
nonfiction_sequences = movies.loc[movies["simple_Genre"] == "Nonfiction", "Overview"].reset_index(drop=True)

for i in tqdm(range(0, len(nonfiction_sequences))):
    sequence = nonfiction_sequences[i]
    predicted_cats += [generate_predictions(sequence, fiction_Genre)]
    actual_cats += ["Nonfiction"]

 38%|███▊      | 603/1597 [00:36<00:59, 16.66it/s]

In [ ]:
predictions_df = pd.DataFrame({"actual_Genre": actual_cats, "predicted_Genre": predicted_cats})

In [ ]:
predictions_df

In [ ]:
predictions_df["correct_prediction"] = (
    np.where(predictions_df["actual_Genre"] == predictions_df["predicted_Genre"], 1, 0)
)

In [ ]:
predictions_df["correct_prediction"].sum() / len(predictions_df)

In [ ]:
isbns = []
predicted_cats = []

missing_cats = movies.loc[movies["simple_Genre"].isna(), ["Title", "Overview"]].reset_index(drop=True)

In [ ]:
for i in tqdm(range(0, len(missing_cats))):
    sequence = missing_cats["Overview"][i]
    predicted_cats += [generate_predictions(sequence, fiction_Genre)]
    isbns += [missing_cats["Title"][i]]

In [ ]:
missing_predicted_df = pd.DataFrame({"Title": isbns, "predicted_Genre": predicted_cats})

In [ ]:
missing_predicted_df

In [ ]:
movies = pd.merge(movies, missing_predicted_df, on="Title", how="left")
movies["simple_Genre"] = np.where(movies["simple_Genre"].isna(), movies["predicted_Genre"], movies["simple_Genre"])
movies = movies.drop(columns = ["predicted_Genre"])

In [ ]:
movies

In [ ]:

movies[movies["Genre"].str.lower().isin([
    "romance",
    "science fiction",
    "scifi",
    "fantasy",
    "horror",
    "mystery",
    "thriller",
    "comedy",
    "crime",
    "historical"
])]

In [ ]:
movies.to_csv("movies_with_Genre.csv", index=False)